# 10 — Avaliacao do Sabia 4 no split de teste

**Objetivo:** Medir F1, precision, recall e acuracia do Sabia 4 (Maritaca AI) em uma amostra balanceada do mesmo `test.parquet` usado pelos demais metodos (TF-IDF, BERTs, ensembles). Objetivo e validacao qualitativa das respostas, nao benchmark robusto.

**Entrada:** `artifacts/splits/test.parquet` (33.258 amostras, 12,61% positivos).

**Protocolo:**
- Prompt zero-shot compartilhado com o notebook 09 (`build_review_prompt`).
- Amostra balanceada: `N_POS=500` positivos (mercado) + `N_NEG=500` negativos (outros), sorteados com seed=42.
- A distribuicao balanceada facilita a leitura de precision/recall/F1, porem NAO reflete a distribuicao de producao (87,4%/12,6%) — os numeros nao sao diretamente comparaveis as Tabelas I/III do artigo.
- Sabia 4 retorna `mercado`/`outros`; convertemos para 0/1 na mesma convencao do projeto.
- Metricas calculadas com `compute_binary_metrics`, McNemar pareado vs os outros metodos nos mesmos indices.

**Saida:** `artifacts/sabia4_test_predictions.csv` (record_id, y_true, y_pred, label_sabia4, justificativa).

**Custo estimado:** 1.000 chamadas x ~1,5s = ~25 min. Tokens: ~1 M input + ~0,05 M output => ~R$ 6 com pricing Maritaca (confirme no painel).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

from economy_classifier.project import SPLITS_DIR, RUNS_DIR
from economy_classifier.evaluation import (
    compute_binary_metrics,
    compute_mcnemar_test,
)
from economy_classifier.ensemble import discover_runs, load_run_predictions
from economy_classifier.llm_review import classify_batch, classify_single
from economy_classifier.visualization import (
    configure_style,
    plot_confusion_matrix,
    save_figure,
)

configure_style()
print('Imports carregados.')

## 1. Configuracao

- `N_POS` / `N_NEG`: tamanho da amostra por classe. Default 500+500 (balanceado).
- `SEED`: fixada em 42 para reproducibilidade.
- `CHECKPOINT_EVERY`: frequencia de salvamento intermediario.

In [ ]:
ARTIFACTS_DIR = Path('../artifacts')
OUTPUT_CSV = ARTIFACTS_DIR / 'sabia4_test_predictions.csv'
CHECKPOINT_CSV = ARTIFACTS_DIR / 'sabia4_test_checkpoint.csv'

MODEL = 'sabia-4'
BASE_URL = 'https://chat.maritaca.ai/api'
TEMPERATURE = 0
SEED = 42

N_POS = 500
N_NEG = 500
CHECKPOINT_EVERY = 50

print(f'Amostra: {N_POS} positivos + {N_NEG} negativos = {N_POS + N_NEG} total')
print(f'Saida: {OUTPUT_CSV}')

## 2. Carregar split de teste e construir amostra balanceada

Sorteio independente por classe, sem reposicao, com seed fixa.

In [ ]:
test = pd.read_parquet(SPLITS_DIR / 'test.parquet')
print(f'Teste completo: {len(test):,} amostras')
print(f'  mercado (1): {(test["label"] == 1).sum():,}')
print(f'  outros  (0): {(test["label"] == 0).sum():,}')

pos_pool = test[test['label'] == 1]
neg_pool = test[test['label'] == 0]
assert len(pos_pool) >= N_POS, f'Positivos insuficientes ({len(pos_pool)} < {N_POS})'
assert len(neg_pool) >= N_NEG, f'Negativos insuficientes ({len(neg_pool)} < {N_NEG})'

pos_sample = pos_pool.sample(n=N_POS, random_state=SEED)
neg_sample = neg_pool.sample(n=N_NEG, random_state=SEED)
sample = pd.concat([pos_sample, neg_sample]).sort_index()

print(f'\nAmostra balanceada: {len(sample):,}')
print(f'  mercado: {(sample["label"] == 1).sum():,}')
print(f'  outros:  {(sample["label"] == 0).sum():,}')

sample_indices = sample.index.tolist()

## 3. Cliente Maritaca AI

Defina `MARITACA_API_KEY` no ambiente antes de rodar (`export MARITACA_API_KEY=...` ou arquivo `.env`). Nao hardcode a chave no notebook.

In [ ]:
os.environ['MARITACA_API_KEY'] = '104367394986916546573_fb195f190de3ba0d'

In [ ]:
api_key = os.environ.get('MARITACA_API_KEY')
assert api_key, 'Defina MARITACA_API_KEY no ambiente antes de rodar.'

client = OpenAI(
    api_key=api_key,
    base_url=BASE_URL,
    max_retries=3,
    timeout=30.0,
)

test_result = classify_single(client, 'Bolsa sobe 2% com alta do dolar', model=MODEL)
assert test_result['label'] in ('mercado', 'outros'), 'API nao retornou label valido'
print(f'Validacao: {test_result["label"]} — API OK.')

## 4. Classificacao com Sabia 4

Checkpoint a cada `CHECKPOINT_EVERY` registros. Retoma automaticamente se `CHECKPOINT_CSV` existir.

In [ ]:
if CHECKPOINT_CSV.exists():
    done_df = pd.read_csv(CHECKPOINT_CSV)
    done_ids = set(done_df['record_id'].astype(int))
    remaining = sample[~sample.index.isin(done_ids)]
    prior_results = done_df.to_dict('records')
    print(f'Retomando: {len(done_ids):,} ja classificados, {len(remaining):,} restantes')
else:
    remaining = sample
    prior_results = []
    print(f'Iniciando classificacao de {len(remaining):,} registros')

texts = remaining['text'].fillna('').tolist()
record_ids = remaining.index.astype(int).tolist()

pbar = tqdm(total=len(texts), desc='Sabia 4')
new_results = classify_batch(
    client, texts, record_ids,
    model=MODEL,
    temperature=TEMPERATURE,
    checkpoint_path=CHECKPOINT_CSV,
    checkpoint_every=CHECKPOINT_EVERY,
    progress_callback=pbar.update,
)
pbar.close()

all_results = prior_results + new_results
print(f'\nClassificacao concluida: {len(all_results):,} registros')

## 5. Montar DataFrame e converter labels

In [ ]:
results_df = pd.DataFrame(all_results).rename(columns={
    'label': 'label_sabia4',
    'justificativa': 'justificativa_sabia4',
    'raw_response': 'raw_response_sabia4',
})
results_df['record_id'] = results_df['record_id'].astype(int)

out = sample.reset_index().rename(columns={'index': 'record_id'}).merge(
    results_df, on='record_id', how='left'
)

# Labels binarias: mercado=1, outros=0. Erros de parsing viram NaN.
label_map = {'mercado': 1, 'outros': 0}
out['y_pred'] = out['label_sabia4'].map(label_map)
out = out.rename(columns={'label': 'y_true'})

n_erros = out['y_pred'].isna().sum()
print(f'Erros de parsing: {n_erros}')
print(f'Distribuicao Sabia 4:')
print(out['label_sabia4'].value_counts())

## 6. Metricas de classificacao

Calculadas apenas sobre registros com predicao valida (exclui erros de parsing).

In [ ]:
valid = out.dropna(subset=['y_pred']).copy()
valid['y_pred'] = valid['y_pred'].astype(int)

metrics = compute_binary_metrics(valid['y_true'].values, valid['y_pred'].values)
metrics['n'] = len(valid)
metrics['n_erros'] = int(n_erros)

print(f"Sabia 4 — metricas na amostra de teste ({len(valid):,} registros validos)")
for k, v in metrics.items():
    print(f'  {k:10s} {v}')

## 7. Matriz de confusao

In [ ]:
cm = pd.crosstab(
    valid['y_true'].map({0: 'outros', 1: 'mercado'}),
    valid['y_pred'].map({0: 'outros', 1: 'mercado'}),
    rownames=['real'],
    colnames=['Sabia 4'],
    margins=True,
    margins_name='Total',
)
print('Matriz de confusao:\n')
print(cm.to_string())

fig_cm = plot_confusion_matrix(
    valid['y_true'].values,
    valid['y_pred'].values,
    title=f'Sabia 4 — Matriz de Confusao (n={len(valid):,})',
)
fig_cm.tight_layout()

## 8. Comparacao com outros metodos nos mesmos indices

Carrega `predictions_test.csv` de cada run, restringe aos indices da amostra e calcula as metricas sobre o mesmo subconjunto que o Sabia 4 viu.

In [ ]:
runs = discover_runs(RUNS_DIR)
valid_ids = set(valid['record_id'].tolist())

rows = [{'method': 'sabia-4', **metrics}]

other_preds = {}
for method, info in sorted(runs.items()):
    tp = load_run_predictions(info['run_dir'], split='test')
    if tp is None:
        continue
    tp = tp[tp['index'].isin(valid_ids)].sort_values('index').reset_index(drop=True)
    if len(tp) == 0:
        continue
    m = compute_binary_metrics(tp['y_true'].values, tp['y_pred'].values)
    m['n'] = len(tp)
    m['n_erros'] = 0
    rows.append({'method': method, **m})
    other_preds[method] = tp

comp_df = pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)
print(f'Comparacao na mesma amostra (n={len(valid):,}):\n')
print(comp_df.to_string(index=False))

## 9. McNemar — Sabia 4 vs demais metodos

Todos os testes sao feitos sobre os mesmos indices da amostra, com alinhamento posicional garantido por ordenacao.

In [ ]:
sabia_ordered = valid.sort_values('record_id').reset_index(drop=True)
y_true = sabia_ordered['y_true'].values
sabia_pred = sabia_ordered['y_pred'].values

rows = []
for method, tp in other_preds.items():
    tp_aligned = tp.set_index('index').loc[sabia_ordered['record_id']].reset_index()
    res = compute_mcnemar_test(y_true, sabia_pred, tp_aligned['y_pred'].values)
    rows.append({'A': 'sabia-4', 'B': method, **res})

mcnemar_df = pd.DataFrame(rows).sort_values('chi2', ascending=False).reset_index(drop=True)
print('McNemar — Sabia 4 vs cada metodo individual:\n')
print(mcnemar_df.to_string(index=False))

## 10. Persistencia

In [ ]:
cols_out = [
    'record_id', 'y_true', 'y_pred', 'label_sabia4',
    'justificativa_sabia4', 'raw_response_sabia4', 'title', 'category',
]
cols_out = [c for c in cols_out if c in out.columns]
out[cols_out].to_csv(OUTPUT_CSV, index=False)
print(f'Predicoes salvas em {OUTPUT_CSV}')

comp_df.to_csv(ARTIFACTS_DIR / 'sabia4_test_comparison.csv', index=False)
mcnemar_df.to_csv(ARTIFACTS_DIR / 'sabia4_test_mcnemar.csv', index=False)
print('Comparacao e McNemar salvos.')

save_figure(fig_cm, ARTIFACTS_DIR / 'figures', 'sabia4_test_confusion_matrix')

if CHECKPOINT_CSV.exists():
    CHECKPOINT_CSV.unlink()
    print('Checkpoint removido.')

## 11. Leitura

- Amostra balanceada (500/500) com prevalencia de 50% — precision e recall sao lidos diretamente, sem o efeito de compressao do desbalanceamento.
- Os numeros **nao** sao comparaveis diretamente aos das Tabelas I/III do artigo (que usam a prevalencia natural de 12,6%). Para comparacao formal, reavalie os outros metodos na mesma amostra (secao 8 faz isso).
- Como o Sabia 4 retorna apenas a classe (sem `y_score` continuo), AUC-ROC nao se aplica.
- A leitura qualitativa das justificativas (coluna `justificativa_sabia4`) e uma segunda camada de validacao — util para identificar padroes de erro do LLM vs erros dos classificadores supervisionados.